# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

I0000 00:00:1775050873.008903 2903067 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775050873.339239 2903067 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775050879.989831 2903067 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['RIUKLQFQNA', 'JJRUZISWIR'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[18,  9, 21, 11, 12, 17,  6, 17, 14,  1],
       [10, 10, 18, 21, 26,  9, 19, 23,  9, 18]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0,  1, 14, 17,  6, 17, 12, 11, 21,  9],
       [ 0, 18,  9, 23, 19,  9, 26, 21, 18, 10]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 1, 14, 17,  6, 17, 12, 11, 21,  9, 18],
       [18,  9, 23, 19,  9, 26, 21, 18, 10, 10]], dtype=int32)>)


I0000 00:00:1775050884.351333 2903067 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46592 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:b3:00.0, compute capability: 8.6


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [ ]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz = 27
        self.hidden = 128
        
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        
        # Attention投影矩阵(用于双线性Attention)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    # 不加@tf.function，避免初始化时的追踪断裂
    def call(self, enc_ids, dec_ids):
        # 1.编码器提取信息
        enc_emb = self.embed_layer(enc_ids)
        enc_out, enc_state = self.encoder(enc_emb)
        
        # 2.解码器生成基础隐状态
        dec_emb = self.embed_layer(dec_ids)
        dec_out, _ = self.decoder(dec_emb, initial_state=[enc_state])
        
        # 3.计算双线性注意力
        # W_enc将作为Key
        W_enc = self.dense_attn(enc_out)    # shape:[batch, enc_len, hidden]
        
        # dec_out作为Query，计算打分->shape:[batch, dec_len, enc_len]
        scores = tf.matmul(dec_out, W_enc, transpose_b=True)
        attn_weights = tf.nn.softmax(scores, axis=-1)
        
        # 根据权重对Value(enc_out)进行加权求和，得到上下文向量
        context = tf.matmul(attn_weights, enc_out)  # shape:[batch, dec_len, hidden]
        
        # 4.融合上下文与解码器信息，输出最终概率
        combined = tf.concat([context, dec_out], axis=-1)
        logits = self.dense(combined)
        return logits
    
    # 修复原代码强行把SimpleRNN状态包装成列表的Bug
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) 
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, enc_state  # SimpleRNN只需要返回单一状态张量即可
    
    def get_next_token(self, x, state, enc_out):
        # 单步嵌入
        x_emb = self.embed_layer(x) 
        
        # RNN细胞单次更新
        dec_out, state_list = self.decoder_cell(x_emb, states=[state])
        next_state = state_list[0]
        
        # 单步Attention计算
        W_enc = self.dense_attn(enc_out) 
        dec_out_exp = tf.expand_dims(dec_out, axis=1) # [batch, 1, hidden]
        
        scores = tf.matmul(dec_out_exp, W_enc, transpose_b=True) # [batch, 1, enc_len]
        attn_weights = tf.nn.softmax(scores, axis=-1)
        
        context = tf.matmul(attn_weights, enc_out) # [batch, 1, hidden]
        context = tf.squeeze(context, axis=1)      # [batch, hidden]
        
        combined = tf.concat([context, dec_out], axis=-1)
        logits = self.dense(combined)
        
        out = tf.argmax(logits, axis=-1)
        return tf.cast(out, tf.int32), next_state

# Loss函数以及训练逻辑

In [4]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [5]:
optimizer = optimizers.Adam(0.001)
model = mySeq2SeqModel()

# 生成一个哑批次提前唤醒模型权重
_, dummy_enc_x, dummy_dec_x, _ = get_batch(2, 20)
model(dummy_enc_x, dummy_dec_x)
print("模型权重已唤醒，开始训练...")

train(model, optimizer, seqlen=20)

模型权重已唤醒，开始训练...
step 0 : loss 3.3009498
step 500 : loss 0.08544328
step 1000 : loss 0.038290177
step 1500 : loss 0.3896746


<tf.Tensor: shape=(), dtype=float32, numpy=0.007469705305993557>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [6]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        # 修复批次大小获取越界的Bug
        b_sz = tf.shape(init_state)[0] 
        
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
            
        out = tf.concat(collect, axis=-1).numpy()
        # 将生成的Index翻译回英文字母
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    return seq == rev_seq_rev

print("\n---正在测试模型逆置能力---")
results = list(zip(*sequence_reversal()))
print("准确率判断:", [is_reverse(*item) for item in results])
print("实际结果比对(预测结果, 原始输入):")
for r in results[:5]:
    print(r)


---正在测试模型逆置能力---
准确率判断: [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
实际结果比对(预测结果, 原始输入):
('IPGNQRLAOWMNKWRPYHXK', 'KXHYPRWKNMWOALRQNGPI')
('ZDGVCDUFSJUJDOMWWEFL', 'LFEWWMODJUJSFUDCVGDZ')
('DUYUTGKOCIGBRSQTNTAO', 'OATNTQSRBGICOKGTUYUD')
('COXNHTFJBDMBLXBEPEXT', 'TXEPEBXLBMDBJFTHNXOC')
('SYDHJNWDFKEVHOLFMYRH', 'HRYMFLOHVEKFDWNJHDYS')
